# 04 ? Frozen internal-test evaluation

**Objective:** Load each completed B0 best checkpoint and save paired per-image outputs.

**Experiment:** Historical Flat versus Shared-Hard baseline reconstruction, seed 42.

**Config:** `configs/experiments/flat/efficientnet_b0.yaml and configs/experiments/shared_hard/efficientnet_b0.yaml`

**Inputs:** Completed run summaries, frozen checkpoints, exact test manifest and raw test images.

**Outputs:** Prediction CSVs with logits/probabilities, endpoint/oracle metrics and hashed evaluation records.

**Mode:** Evaluation only; no optimization or checkpoint selection. Every input is loaded from disk; no other notebook's kernel state is required.

Use **Restart Kernel ? Run All**. Real training is disabled until the build handover is reviewed.


In [2]:
import os
from pathlib import Path

os.environ["SKIN_CANCER_DATA_ROOT"] = "/home/m-sazzad-h/skin-cancer-work"
DATA_ROOT = Path(os.environ["SKIN_CANCER_DATA_ROOT"])

print("Dataset root:", DATA_ROOT.resolve())
print("data/raw exists:", (DATA_ROOT / "data/raw").exists())

Dataset root: /home/m-sazzad-h/skin-cancer-work
data/raw exists: True


In [3]:
from pathlib import Path
import sys, os
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "configs/protocol.yaml").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from IPython.display import display, Markdown
from src.config import load_config
DATA_ROOT = os.environ.get("SKIN_CANCER_DATA_ROOT")
print("Dataset root:", Path(DATA_ROOT).expanduser().resolve() if DATA_ROOT else "NOT CONFIGURED ? set SKIN_CANCER_DATA_ROOT before image verification/training")


Dataset root: /home/m-sazzad-h/skin-cancer-work


## Explicit frozen checkpoint inputs

The historical internal test is a consumed reproduction benchmark. Do not tune against its results. Oracle gating uses truth only as a diagnostic. Stage-2 logits are retained for every image; Stage-2 truth is missing for true NM.

In [4]:
RUN_EVALUATION = True
DEVICE = "cuda"
RUN_IDS = ["flat_efficientnet_b0_seed42", "shared_hard_efficientnet_b0_seed42"]
from src.evaluation import evaluate
from src.statistics import load_evaluation
for run_id in RUN_IDS:
    if RUN_EVALUATION:
        display(evaluate(run_id, data_root=DATA_ROOT, device=DEVICE))
    elif (ROOT/"experiments/evaluations"/f"{run_id}_test"/"evaluation.json").exists():
        metadata, predictions = load_evaluation(f"{run_id}_test")
        display(metadata); display(predictions.head())
    else:
        print(run_id, "evaluation NOT RUN; complete and freeze training first")

Dataset root: /home/m-sazzad-h/skin-cancer-work


{'run_id': 'flat_efficientnet_b0_seed42',
 'architecture': 'efficientnet_b0',
 'system_type': 'flat',
 'split': 'test',
 'sample_count': 3668,
 'prediction_path': 'results/predictions/flat_efficientnet_b0_seed42_test.csv',
 'prediction_sha256': 'c62b095639c99dd2228dbc8e5c3046e65fcb44f7bafba303ed461879bc5980bb',
 'checkpoint': {'path': 'models/checkpoints/flat_efficientnet_b0_seed42/epoch_005_dd7f1ae1/best.pt',
  'sha256': 'fe13734dcfee1bf945b4651fcb819c957506f1bacf7312c18520a5b837594b8e'},
 'manifest_hashes': {'isic': '4aff7b36a942ba19144c01d22910f356534d45603b00844d242e4021892fc906',
  'stage3': '1f2296af7caf2a2520ab8f288a026e42c475d35d66ac9f488e05947a2a843895'},
 'config_hash': 'e9bb47a6117aaa585a978e849d78d8c98df340643ccc06ba688f14cfa3da92c0',
 'environment': {'python': '3.11.9 (main, Aug 14 2024, 05:07:28) [Clang 18.1.8 ]',
  'torch': '2.13.0+cu126',
  'torchvision': '0.28.0+cu126',
  'numpy': '2.4.6',
  'cuda': '12.6',
  'cudnn': 91002,
  'gpu': 'Tesla T4',
  'os': 'Linux-6.17.0-1

Dataset root: /home/m-sazzad-h/skin-cancer-work


{'run_id': 'shared_hard_efficientnet_b0_seed42',
 'architecture': 'efficientnet_b0',
 'system_type': 'shared_hard',
 'split': 'test',
 'sample_count': 3668,
 'prediction_path': 'results/predictions/shared_hard_efficientnet_b0_seed42_test.csv',
 'prediction_sha256': '284b7fe6b8c4781cbc03fc7888702bea27eca4aa94928c18c398ddf34c6a7b2b',
 'checkpoint': {'path': 'models/checkpoints/shared_hard_efficientnet_b0_seed42/epoch_013_da2db014/best.pt',
  'sha256': 'b6feaf58346f1e61600c09362bda948c7950a8f97b71098eff3a79c33c998c9a'},
 'manifest_hashes': {'isic': '4aff7b36a942ba19144c01d22910f356534d45603b00844d242e4021892fc906',
  'stage3': '1f2296af7caf2a2520ab8f288a026e42c475d35d66ac9f488e05947a2a843895'},
 'config_hash': '80870a870afb648bef498a6baf95a3a53198eefd160085c9731c65407b01f3c1',
 'environment': {'python': '3.11.9 (main, Aug 14 2024, 05:07:28) [Clang 18.1.8 ]',
  'torch': '2.13.0+cu126',
  'torchvision': '0.28.0+cu126',
  'numpy': '2.4.6',
  'cuda': '12.6',
  'cudnn': 91002,
  'gpu': 'Tesla 

## Summary and next step

Review the status and metrics displayed above. Missing artifacts mean **not run**, never a successful reproduction. Generated artifacts are listed in the output cells; scientific results remain separate from historical reference values.

**Next:** `05_flat_vs_hierarchical_comparison.ipynb`. Preserve completed run directories, prediction files and checkpoint backups before continuing.
